In [ ]:
import os
import cv2
import numpy as np
import random
import matplotlib.pyplot as plt
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision


class Config:
    def __init__(self):
        self.root_dir = "/content/dataset1/"
        self.model_path = "/content/blaze_face_short_range.tflite"

        self.detection_threshold = 0.5

        self.head_width_scale = 1.30
        self.head_height_scale = 1.50
        self.head_vertical_shift = 0.12

        self.num_random_frames_per_video = 2
        self.random_seed = 42



class MediaPipeDetector:
    def __init__(self, model_path, threshold=0.5):
        base_options = python.BaseOptions(model_asset_path=model_path)
        options = vision.FaceDetectorOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.IMAGE,
            min_detection_confidence=0.2,
        )
        self.detector = vision.FaceDetector.create_from_options(options)
        self.threshold = threshold

    def detect(self, frame_bgr):
        try:
            if len(frame_bgr.shape) == 2:
                frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_GRAY2RGB)
            else:
                frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
            result = self.detector.detect(mp_image)
        except Exception:
            return None

        if not result.detections:
            return None

        best_face = None
        best_area = -1

        for det in result.detections:
            score = float(det.categories[0].score)
            if score < self.threshold:
                continue

            bbox = det.bounding_box
            x1 = bbox.origin_x
            y1 = bbox.origin_y
            x2 = x1 + bbox.width
            y2 = y1 + bbox.height

            area = (x2 - x1) * (y2 - y1)
            if area > best_area:
                best_area = area
                best_face = {"bbox": [x1, y1, x2, y2]}

        return best_face

    def close(self):
        self.detector.close()



def compute_head_box_from_face(face, cfg):
    x1, y1, x2, y2 = face["bbox"]

    face_w = x2 - x1
    face_h = y2 - y1

    cx = x1 + face_w / 2
    cy = y1 + face_h / 2

    cy -= cfg.head_vertical_shift * face_h

    crop_w = int(face_w * cfg.head_width_scale)
    crop_h = int(face_h * cfg.head_height_scale)

    x1 = int(cx - crop_w / 2)
    y1 = int(cy - crop_h / 2)
    x2 = x1 + crop_w
    y2 = y1 + crop_h

    return x1, y1, x2, y2


def crop_with_reflect_padding(img, x1, y1, x2, y2):
    h, w = img.shape[:2]

    pad_l = max(0, -x1)
    pad_t = max(0, -y1)
    pad_r = max(0, x2 - w)
    pad_b = max(0, y2 - h)

    if pad_l or pad_t or pad_r or pad_b:
        img = cv2.copyMakeBorder(
            img,
            pad_t, pad_b, pad_l, pad_r,
            borderType=cv2.BORDER_REFLECT
        )

    x1 += pad_l
    x2 += pad_l
    y1 += pad_t
    y2 += pad_t

    return img[y1:y2, x1:x2]


def extract_crop(frame, box):
    if box is None:
        return None

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return crop_with_reflect_padding(gray, *box)



def collect_videos(root):
    items = []

    for p in os.listdir(root):
        p_path = os.path.join(root, p)
        if not os.path.isdir(p_path):
            continue

        for c in os.listdir(p_path):
            c_path = os.path.join(p_path, c)
            if not os.path.isdir(c_path):
                continue

            for f in os.listdir(c_path):
                if f.endswith(".avi"):
                    items.append({
                        "person": p,
                        "condition": c,
                        "video": f,
                        "path": os.path.join(c_path, f)
                    })

    return items


def sample_frames(video_path, n, rng):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    idxs = sorted(rng.sample(range(total), min(n, total)))

    frames = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if ret:
            frames.append((i, frame))

    cap.release()
    return frames



def preview_head(cfg):
    rng = random.Random(cfg.random_seed)
    detector = MediaPipeDetector(cfg.model_path, cfg.detection_threshold)

    videos = collect_videos(cfg.root_dir)
    results = []

    for item in videos:
        frames = sample_frames(item["path"], cfg.num_random_frames_per_video, rng)

        for idx, frame in frames:
            face = detector.detect(frame)

            if face is None:
                results.append((frame, None, "no_face"))
                continue

            box = compute_head_box_from_face(face, cfg)
            crop = extract_crop(frame, box)

            vis = frame.copy()
            x1, y1, x2, y2 = box
            cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)

            results.append((vis, crop, "ok"))

    detector.close()


    rows = len(results)
    plt.figure(figsize=(10, 4 * rows))

    i = 1
    for vis, crop, status in results:
        plt.subplot(rows, 2, i)
        plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
        plt.title(status)
        plt.axis("off")
        i += 1

        plt.subplot(rows, 2, i)
        if crop is not None:
            plt.imshow(crop, cmap="gray")
        else:
            plt.imshow(np.full((50, 50), 128), cmap="gray")
        plt.axis("off")
        i += 1

    plt.tight_layout()
    plt.show()



if __name__ == "__main__":
    cfg = Config()
    preview_head(cfg)